In [1]:
DATA_DIR = "../scratch/data"

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [9]:
snv_annotations = pd.read_csv("../data/williams/signatures_dataset/DLPextra/SNV/snv_annotations.csv.gz")

/var/folders/nz/jn74t1g51q10qyclfkkr8cgm0000gq/T/ipykernel_85495/3023799263.py:1: DtypeWarning: Columns (0,15,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  snv_annotations = pd.read_csv("../data/williams/signatures_dataset/DLPextra/SNV/snv_annotations.csv.gz")


In [10]:
ov2295_annotations = snv_annotations[snv_annotations['patient'] == 'OV2295']
ov2295_annotations

,chr,pos,ref,alt,max_strelka_score,max_museq_score,mappability,is_cosmic,gene_name,effect,effect_impact,amino_acid_change,is_dbsnp,tri_nucleotide_context,patient,isabl_patient_id,isabl_sample_id,isabl_aliquot_id
859961,1,1142720,G,C,58,0.90,1.0,0,TNFRSF18,UPSTREAM,MODIFIER,NaN,0,GGA,OV2295,NaN,NaN,NaN
859962,1,1147615,A,G,24,0.92,1.0,0,TNFRSF4,NEXT_PROT[glycosylation-site:N-linked__GlcNAc....,MODERATE,NaN,0,CAG,OV2295,NaN,NaN,NaN
859963,1,1355571,T,C,52,0.90,1.0,0,ANKRD65,NON_SYNONYMOUS_CODING,MODERATE,D204G,0,GTC,OV2295,NaN,NaN,NaN
859964,1,1472034,G,C,97,0.92,1.0,0,ATAD3A,DOWNSTREAM,MODIFIER,NaN,0,AGC,OV2295,NaN,NaN,NaN
859965,1,1951177,T,C,44,0.93,1.0,0,NaN,MOTIF[MA0138.2:Nrsf],MODIFIER,NaN,0,GTC,OV2295,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
894603,X,153762606,G,A,27,0.97,1.0,0,G6PD,SYNONYMOUS_CODING,LOW,Y197,0,GGT,OV2295,NaN,NaN,NaN
894604,X,154012317,G,A,151,0.93,1.0,0,MPP1,NON_SYNONYMOUS_CODING,MODERATE,T284I,0,GGT,OV2295,NaN,NaN,NaN
894605,X,154299695,A,C,36,0.90,1.0,0,BRCC3,UTR_5_PRIME,MODIFIER,NaN,0,CAG,OV2295,NaN,NaN,NaN
894606,X,154563943,T,A,313,0.99,1.0,0,CLIC2,UTR_5_PRIME,MODIFIER,NaN,0,CTC,OV2295,NaN,NaN,NaN


In [3]:
df = pd.read_csv(f"{DATA_DIR}/ov2295_clone_cn.csv.gz")
df

,clone_id,chr,start,end,copy,total_cn,minor_cn,major_cn
0,E,1,1,500000,NaN,4,0,4
1,E,1,500001,1000000,NaN,4,0,4
2,E,1,1000001,1500000,4.906717,4,0,4
3,E,1,1500001,2000000,3.793922,4,0,4
4,E,1,2000001,2500000,5.130191,4,0,4
...,...,...,...,...,...,...,...,...
55849,F,Y,57000001,57500000,NaN,0,0,0
55850,F,Y,57500001,58000000,NaN,0,0,0
55851,F,Y,58000001,58500000,NaN,0,0,0
55852,F,Y,58500001,59000000,NaN,0,0,0


In [4]:
df["is_loh"] = (df["minor_cn"] == 0)


In [5]:
clone_count = df["clone_id"].nunique()

common_loh = (
    df[df["is_loh"]]
    .groupby(["chr", "start", "end"])
    .agg(n_clones=("clone_id", "nunique"))
    .reset_index()
)

common_loh = common_loh[common_loh["n_clones"] == clone_count]


In [6]:
common_loh

,chr,start,end,n_clones
0,1,1,500000,9
1,1,500001,1000000,9
2,1,1000001,1500000,9
3,1,1500001,2000000,9
4,1,2000001,2500000,9
...,...,...,...,...
3492,Y,57000001,57500000,9
3493,Y,57500001,58000000,9
3494,Y,58000001,58500000,9
3495,Y,58500001,59000000,9


In [7]:
common_index = common_loh.set_index(["chr","start","end"]).index

df["is_common"] = (
    df.set_index(["chr","start","end"]).index.isin(common_index)
)

df["loh_not_common"] = df["is_loh"] & (~df["is_common"])


In [134]:
group = ["E"]

group_loh = (
    df[df["clone_id"].isin(group) & df["loh_not_common"]]
    .groupby(["chr","start","end"])
    .agg(n=("clone_id","nunique"))
    .reset_index()
)

group_loh

,chr,start,end,n
0,1,110000001,110500000,1
1,1,110500001,111000000,1
2,1,111000001,111500000,1
3,1,111500001,112000000,1
4,1,112000001,112500000,1
...,...,...,...,...
475,7,103000001,103500000,1
476,7,103500001,104000000,1
477,8,3500001,4000000,1
478,9,70000001,70500000,1


In [135]:
ov2295_annotations["chr"] = ov2295_annotations["chr"].astype(str)
group_loh["chr"] = group_loh["chr"].astype(str)

/var/folders/nz/jn74t1g51q10qyclfkkr8cgm0000gq/T/ipykernel_85495/367932570.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ov2295_annotations["chr"] = ov2295_annotations["chr"].astype(str)


In [136]:
merged = ov2295_annotations.merge(
    group_loh,
    on="chr",
    how="inner"
)


In [137]:
mut_in_loh = merged[
    (merged["pos"] >= merged["start"]) &
    (merged["pos"] <= merged["end"])
]


In [138]:
mut_in_loh = mut_in_loh.drop_duplicates(
    subset=["chr", "pos", "ref", "alt"]
)


In [139]:
mut_in_loh

,chr,pos,ref,alt,max_strelka_score,max_museq_score,mappability,is_cosmic,gene_name,effect,...,amino_acid_change,is_dbsnp,tri_nucleotide_context,patient,isabl_patient_id,isabl_sample_id,isabl_aliquot_id,start,end,n
4496,1,110192801,G,A,71,0.95,1.0,0,RPL7P8,UPSTREAM,...,NaN,1,TGA,OV2295,NaN,NaN,NaN,110000001,110500000,1
4513,1,110742481,A,G,23,0.98,1.0,0,SLC6A17,UTR_3_PRIME,...,NaN,0,TAG,OV2295,NaN,NaN,NaN,110500001,111000000,1
4529,1,110886211,A,G,67,0.99,1.0,0,RP5-1074L1.1,UPSTREAM,...,NaN,0,AAT,OV2295,NaN,NaN,NaN,110500001,111000000,1
4548,1,112186444,A,T,142,0.99,1.0,0,KRT18P57,DOWNSTREAM,...,NaN,0,CAC,OV2295,NaN,NaN,NaN,112000001,112500000,1
4564,1,112430335,T,G,207,0.97,1.0,0,KCND3,INTRON,...,NaN,0,CTG,OV2295,NaN,NaN,NaN,112000001,112500000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
962517,7,101847546,T,A,47,0.90,1.0,0,CUX1,INTRON,...,NaN,0,GTG,OV2295,NaN,NaN,NaN,101500001,102000000,1
962614,7,101847547,G,C,50,0.93,1.0,0,CUX1,INTRON,...,NaN,0,TGG,OV2295,NaN,NaN,NaN,101500001,102000000,1
962711,7,101923572,G,A,60,0.96,1.0,0,SH2B2,UPSTREAM,...,NaN,0,AGC,OV2295,NaN,NaN,NaN,101500001,102000000,1
962909,7,103645468,C,T,41,0.97,1.0,0,NaN,INTERGENIC,...,NaN,0,TCA,OV2295,NaN,NaN,NaN,103500001,104000000,1


In [140]:
cbio = pd.read_csv("../data/laks/Mutated_Genes.txt", sep="\t")
cbio["frequency"] = cbio["Freq"].str.rstrip("%").astype(float)
cbio = cbio.sort_values("frequency", ascending=False)
cbio

,Gene,MutSig(Q-value),# Mut,#,Profiled Samples,Freq,Is Cancer Gene (source: OncoKB),frequency
5978,TP53,NaN,374,369,405,91.1%,Yes,91.1
1195,TTN,NaN,120,96,405,23.7%,No,23.7
1690,CSMD3,NaN,37,36,405,8.9%,No,8.9
4539,USH2A,NaN,32,31,405,7.7%,No,7.7
7891,NF1,NaN,30,29,405,7.2%,Yes,7.2
...,...,...,...,...,...,...,...,...
7340,INTS4,NaN,1,1,405,0.2%,No,0.2
3131,SYBU,NaN,1,1,405,0.2%,No,0.2
3132,CISH,NaN,1,1,405,0.2%,No,0.2
7337,INTS5,NaN,1,1,405,0.2%,No,0.2


In [141]:
cbio_with_freq_gt = cbio[cbio['frequency'] >= 0]
cbio_with_freq_gt

,Gene,MutSig(Q-value),# Mut,#,Profiled Samples,Freq,Is Cancer Gene (source: OncoKB),frequency
5978,TP53,NaN,374,369,405,91.1%,Yes,91.1
1195,TTN,NaN,120,96,405,23.7%,No,23.7
1690,CSMD3,NaN,37,36,405,8.9%,No,8.9
4539,USH2A,NaN,32,31,405,7.7%,No,7.7
7891,NF1,NaN,30,29,405,7.2%,Yes,7.2
...,...,...,...,...,...,...,...,...
7340,INTS4,NaN,1,1,405,0.2%,No,0.2
3131,SYBU,NaN,1,1,405,0.2%,No,0.2
3132,CISH,NaN,1,1,405,0.2%,No,0.2
7337,INTS5,NaN,1,1,405,0.2%,No,0.2


In [142]:
mut_in_loh["gene_name"] = mut_in_loh["gene_name"].str.strip().str.upper()
cbio_with_freq_gt["Gene"] = cbio_with_freq_gt["Gene"].str.strip().str.upper()

In [143]:
mut_loh_cbio_gene = mut_in_loh.merge(
    cbio_with_freq_gt,
    left_on="gene_name",
    right_on="Gene",
    how="inner"
)
mut_loh_cbio_gene

,chr,pos,ref,alt,max_strelka_score,max_museq_score,mappability,is_cosmic,gene_name,effect,...,end,n,Gene,MutSig(Q-value),# Mut,#,Profiled Samples,Freq,Is Cancer Gene (source: OncoKB),frequency
0,1,110742481,A,G,23,0.98,1.0,0,SLC6A17,UTR_3_PRIME,...,111000000,1,SLC6A17,NaN,3,3,405,0.7%,No,0.7
1,1,112430335,T,G,207,0.97,1.0,0,KCND3,INTRON,...,112500000,1,KCND3,NaN,3,3,405,0.7%,No,0.7
2,1,113257851,A,T,24,0.92,1.0,0,PPM1J,NON_SYNONYMOUS_CODING,...,113500000,1,PPM1J,NaN,1,1,405,0.2%,No,0.2
3,1,114157531,T,G,35,0.92,1.0,0,MAGI3,INTRON,...,114500000,1,MAGI3,NaN,7,7,405,1.7%,No,1.7
4,1,114157553,G,C,91,0.95,1.0,0,MAGI3,INTRON,...,114500000,1,MAGI3,NaN,7,7,405,1.7%,No,1.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
697,7,101505677,T,C,34,0.92,1.0,0,CUX1,INTRON,...,102000000,1,CUX1,NaN,9,8,405,2.0%,Yes,2.0
698,7,101723478,A,G,63,0.94,1.0,0,CUX1,INTRON,...,102000000,1,CUX1,NaN,9,8,405,2.0%,Yes,2.0
699,7,101847546,T,A,47,0.90,1.0,0,CUX1,INTRON,...,102000000,1,CUX1,NaN,9,8,405,2.0%,Yes,2.0
700,7,101847547,G,C,50,0.93,1.0,0,CUX1,INTRON,...,102000000,1,CUX1,NaN,9,8,405,2.0%,Yes,2.0


In [144]:
mut_loh_cbio_gene.to_csv('scratch.csv')

In [85]:
merged = df.merge(
    df,
    on=['chr', 'start', 'end'],
    suffixes=('_1', '_2')
)
merged = merged[merged['clone_id_1'] != merged['clone_id_2']]

In [86]:
merged

,clone_id_1,chr,start,end,copy_1,total_cn_1,minor_cn_1,major_cn_1,is_loh_1,is_common_1,loh_not_common_1,clone_id_2,copy_2,total_cn_2,minor_cn_2,major_cn_2,is_loh_2,is_common_2,loh_not_common_2
1,E,1,1,500000,NaN,4,0,4,True,True,False,C,NaN,2,0,2,True,True,False
2,E,1,1,500000,NaN,4,0,4,True,True,False,G,NaN,2,0,2,True,True,False
3,E,1,1,500000,NaN,4,0,4,True,True,False,D,NaN,2,0,2,True,True,False
4,E,1,1,500000,NaN,4,0,4,True,True,False,A,NaN,2,0,2,True,True,False
5,E,1,1,500000,NaN,4,0,4,True,True,False,B,NaN,2,0,2,True,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
502680,F,Y,59000001,59500000,NaN,0,0,0,True,True,False,D,NaN,0,0,0,True,True,False
502681,F,Y,59000001,59500000,NaN,0,0,0,True,True,False,A,NaN,0,0,0,True,True,False
502682,F,Y,59000001,59500000,NaN,0,0,0,True,True,False,B,NaN,0,0,0,True,True,False
502683,F,Y,59000001,59500000,NaN,0,0,0,True,True,False,I,NaN,0,0,0,True,True,False


In [87]:
clone_1 = 'A'
clone_2 = 'B'

In [88]:
merged[(merged['clone_id_1'] == clone_1) & (merged['clone_id_2'] == clone_2) & (merged['minor_cn_1'] > 0) & (merged['minor_cn_2'] == 0)]

,clone_id_1,chr,start,end,copy_1,total_cn_1,minor_cn_1,major_cn_1,is_loh_1,is_common_1,loh_not_common_1,clone_id_2,copy_2,total_cn_2,minor_cn_2,major_cn_2,is_loh_2,is_common_2,loh_not_common_2
229649,A,10,96500001,97000000,1.574454,2,1,1,False,False,False,B,1.690818,2,0,2,True,False,True
237650,A,14,20000001,20500000,1.515784,2,1,1,False,False,False,B,1.517854,1,0,1,True,False,True
258503,A,4,52500001,53000000,NaN,2,1,1,False,False,False,B,NaN,2,0,2,True,False,True


In [89]:
merged[(merged['clone_id_1'] == clone_1) & (merged['clone_id_2'] == clone_2) & (merged['major_cn_1'] > 0) & (merged['major_cn_2'] == 0)]

,clone_id_1,chr,start,end,copy_1,total_cn_1,minor_cn_1,major_cn_1,is_loh_1,is_common_1,loh_not_common_1,clone_id_2,copy_2,total_cn_2,minor_cn_2,major_cn_2,is_loh_2,is_common_2,loh_not_common_2
